In [32]:
#imports
import pandas as pd
import numpy as np


In [33]:
### Men's Massey Ordinals

#data
massey_ordinals_m = pd.read_csv("data_2026/MMasseyOrdinals.csv")

#pre tournament rankings
pre_tournament_massey_m = massey_ordinals_m.query("RankingDayNum == 133")

#good subset of rankings
#good_ratings_m = ["POM", "EBP", "MAS", "TRK", "HAS"]
#g_pre_tournament_massey_m = pre_tournament_massey_m.query("Season >= 2016").loc[lambda df: df["SystemName"].isin(good_ratings_m)]

#group by and summarize
massey_m = pre_tournament_massey_m.groupby(["Season", "TeamID"]).agg(avg_massey_rank=("OrdinalRank", "mean")).reset_index()


In [34]:
### Men's AP Rankings

#data
massey_ordinals_m = pd.read_csv("data_2026/MMasseyOrdinals.csv")

#preseason rankings
preseason_ap_m = massey_ordinals_m.query("SystemName == 'AP'") 
preseason_ap_m = preseason_ap_m[preseason_ap_m['RankingDayNum'] == preseason_ap_m.groupby('Season')['RankingDayNum'].transform('min')]
preseason_ap_m = preseason_ap_m.rename({'OrdinalRank': 'ap_preseason_rank'}, axis='columns')
preseason_ap_m = preseason_ap_m[['Season', 'TeamID', 'ap_preseason_rank']]

#pretournament ratings
pre_tournament_ap_m = massey_ordinals_m.query("SystemName == 'AP' & RankingDayNum == 133")
pre_tournament_ap_m = pre_tournament_ap_m.rename({'OrdinalRank': 'ap_pretournament_rank'}, axis='columns')
pre_tournament_ap_m = pre_tournament_ap_m[['Season', 'TeamID', 'ap_pretournament_rank']]

#combine
ap_m = pd.merge(preseason_ap_m, pre_tournament_ap_m, how="outer", on=["Season", "TeamID"])


In [35]:
### Get Stats Function

def get_stats(reg_stats):

    #Remove Loc, which causes problems
    reg_stats = reg_stats.drop(columns = 'WLoc')

    w_reg_stats = reg_stats.copy()
    l_reg_stats = reg_stats.copy()

    #For games team is winner
    w_reg_stats.columns = w_reg_stats.columns.str.replace(r'^L', 'opp_', regex=True)
    w_reg_stats.columns = w_reg_stats.columns.str.replace(r'^W', '', regex=True)

    #For games team is loser
    l_reg_stats.columns = l_reg_stats.columns.str.replace(r'^W', 'opp_', regex=True)
    l_reg_stats.columns = l_reg_stats.columns.str.replace(r'^L', '', regex=True)

    #combine the data
    reg_stats_pg = pd.concat([w_reg_stats, l_reg_stats])

    #per game percentages (need for tempo adjusted average percentages)
    reg_stats_pg = (
        reg_stats_pg.assign(margin = reg_stats_pg['Score'] - reg_stats_pg['opp_Score'])
            .assign(poss = lambda x: x['FGA'] - x['OR'] + x['TO'] + 0.475*x['FTA'])
            .assign(opp_poss = lambda x: x['opp_FGA'] - x['opp_OR'] + x['opp_TO'] + 0.475*x['opp_FTA'])
            .assign(eff=lambda x: x['Score'] / x['poss'])
            .assign(opp_eff=lambda x: x['opp_Score'] / x['opp_poss'])

            .assign(fg_per=lambda x: x['FGM'] / x['FGA'])
            .assign(fg_a_per=lambda x: x['FGA'] / x['poss'])
            .assign(thr_a_per=lambda x: x['FGA3'] / x['poss'])
            .assign(to_per=lambda x: x['TO'] / x['poss'])
            .assign(blk_per=lambda x: x['Blk'] / x['opp_FGA'])
            .assign(foul_rec_per=lambda x: x['opp_PF'] / x['poss'])
            .assign(foul_per=lambda x: x['PF'] / x['opp_poss'])
            .assign(or_per=lambda x: x['OR'] / (x['OR'] + x['opp_DR']))
            .assign(dr_per=lambda x: x['DR'] / (x['DR'] + x['opp_OR']))
        
            .assign(opp_fg_a_per=lambda x: x['opp_FGA'] / x['opp_poss'])
            .assign(opp_fg_per=lambda x: x['opp_FGM'] / x['opp_FGA'])
            .assign(opp_to_per=lambda x: x['opp_TO'] / x['opp_poss'])
    )

    f_reg_stats = reg_stats_pg.groupby(["Season", "TeamID"]).agg(
        avg_score=("Score", "mean"),
        avg_opp_score=("opp_Score", "mean"),
        avg_margin = ("margin", "mean"),
        avg_poss = ("poss", "mean"),
        avg_eff = ("eff", "mean"),
        avg_opp_eff = ("opp_eff", "mean"),
        avg_fg_per=("fg_per", "mean"),
        avg_m3=("FGM3", "mean"),
        avg_a3=("FGA3", "mean"),#
        avg_ftm=("FTM", "mean"),
        avg_fta=("FTA", "mean"),#
        avg_fg_a_per=("fg_a_per", "mean"),
        avg_thr_a_per=("thr_a_per", "mean"),
        avg_to_per=("to_per", "mean"),
        avg_blk_per=("blk_per", "mean"),
        avg_foul_rec_per=("foul_rec_per", "mean"),
        avg_foul_per=("foul_per", "mean"),
        avg_or_per=("or_per", "mean"),
        avg_dr_per=("dr_per", "mean"),
        avg_opp_fg_a_per=("opp_fg_a_per", "mean"),
        avg_opp_fg_per=("opp_fg_per", "mean"),
        avg_opp_to_per=("opp_to_per", "mean")
    ).reset_index()

    f_reg_stats = (
        f_reg_stats.assign(avg_thr_per = f_reg_stats['avg_m3'] / f_reg_stats['avg_a3'])
        .assign(ft_per=lambda x: x['avg_ftm'] / x['avg_fta'])
    )

    f_reg_stats = f_reg_stats.drop(columns = ['avg_a3', 'avg_m3', 'avg_ftm', 'avg_fta'])

    return f_reg_stats

In [36]:
### Get Stats

#data
reg_stats_m = pd.read_csv("data_2026/MRegularSeasonDetailedResults.csv")
reg_stats_w = pd.read_csv("data_2026/WRegularSeasonDetailedResults.csv")

#men's
f_reg_stats_m = get_stats(reg_stats_m)

#women's
f_reg_stats_w = get_stats(reg_stats_w)


#f_reg_stats_m.sort_values('avg_eff', ascending = False).head()

In [37]:
### Seed
seed_m = pd.read_csv("data_2026/MNCAATourneySeeds.csv")
seed_w = pd.read_csv("data_2026/WNCAATourneySeeds.csv")

seed_m = seed_m.assign(seed = seed_m['Seed'].str[1:3])
seed_w = seed_w.assign(seed = seed_w['Seed'].str[1:3])

seed_m = seed_m[['Season', 'TeamID', 'seed']]
seed_w = seed_w[['Season', 'TeamID', 'seed']]

seed_m.head()

,Season,TeamID,seed
0,1985,1207,01
1,1985,1210,02
2,1985,1228,03
3,1985,1260,04
4,1985,1374,05


In [38]:
### Combine Data

#Men's
combined_m = (
    seed_m
    .merge(ap_m, on=["Season", "TeamID"], how="outer")
    .merge(massey_m, on=["Season", "TeamID"], how="outer")
    .merge(f_reg_stats_m, on=["Season", "TeamID"], how="outer")
)

#Women's
combined_w = (
    seed_w
    .merge(f_reg_stats_w, on=["Season", "TeamID"], how="outer")
)

In [ ]:
### Get results

def get_results(tourney):
    tourney = (tourney.assign(
            team_A = tourney[['WTeamID','LTeamID']].min(axis=1),
            team_B = tourney[['WTeamID','LTeamID']].max(axis=1)
        ).assign(
            result = lambda x: (x['WTeamID'] == x['team_A']).astype(int),
            score_A = lambda x: x['result']*x['WScore'] + (1 - x['result'])*x['LScore'],
            score_B = lambda x: x['result']*x['LScore'] + (1 - x['result'])*x['WScore'],
            margin = lambda x: (x['score_A'] - x['score_B'])
        )
    )

    tourney = tourney[['Season', 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'margin', 'result', 'NumOT']]
    
    return tourney

def get_dup_results(tourney):
    tourney = (tourney.assign(
            team_A = tourney[['WTeamID','LTeamID']].max(axis=1),
            team_B = tourney[['WTeamID','LTeamID']].min(axis=1)
        ).assign(
            result = lambda x: (x['WTeamID'] == x['team_A']).astype(int),
            score_A = lambda x: x['result']*x['WScore'] + (1 - x['result'])*x['LScore'],
            score_B = lambda x: x['result']*x['LScore'] + (1 - x['result'])*x['WScore'],
            margin = lambda x: (x['score_A'] - x['score_B'])
        )
    )

    tourney = tourney[['Season', 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'margin', 'result', 'NumOT']]
    
    return tourney


#Men's
tourney_m = pd.read_csv("data_2026/MNCAATourneyCompactResults.csv")
tourney_results_og_m = get_results(tourney_m)
tourney_results_dup_m = get_dup_results(tourney_m)
tourney_results_m = pd.concat([tourney_results_og_m, tourney_results_dup_m], ignore_index=True)

#Women's
tourney_w = pd.read_csv("data_2026/WNCAATourneyCompactResults.csv")
tourney_results_og_w = get_results(tourney_w)
tourney_results_dup_w = get_dup_results(tourney_w)
tourney_results_w = pd.concat([tourney_results_og_w, tourney_results_dup_w], ignore_index=True)


   Season  DayNum  team_A  team_B  score_A  score_B  margin  result  NumOT
0    1985     136    1116    1234       63       54       9       1      0
1    1985     136    1120    1345       59       58       1       1      0
2    1985     136    1207    1250       68       43      25       1      0
3    1985     136    1229    1425       58       55       3       1      0
4    1985     136    1242    1325       49       38      11       1      0
   Season  TeamID seed  ap_preseason_rank  ap_pretournament_rank  \
0    1985    1104   07                NaN                    NaN   
1    1985    1112   10                NaN                    NaN   
2    1985    1116   09                NaN                    NaN   
3    1985    1120   11                NaN                    NaN   
4    1985    1130   11                NaN                    NaN   

   avg_massey_rank  avg_score  avg_opp_score  avg_margin  avg_poss  ...  \
0              NaN        NaN            NaN         NaN       NaN

In [40]:
# Combine Data with Results

#function 
def combine_data_results(combined, tourney_results):
    team_A_data = combined.rename(
        columns={c: f"{c}_A" for c in combined.columns if c not in ['Season', 'TeamID']}
    )

    team_B_data = combined.rename(
        columns={c: f"{c}_B" for c in combined.columns if c not in ['Season', 'TeamID']}
    )

    full_data = (tourney_results
        .merge(team_A_data, left_on=['Season', 'team_A'], right_on=['Season', 'TeamID'], how="left")
        .drop(columns='TeamID')
        .merge(team_B_data, left_on=['Season', 'team_B'], right_on=['Season', 'TeamID'], how="left")
        .drop(columns='TeamID')
    )

    return full_data


#Men's
full_data_m = combine_data_results(combined_m, tourney_results_m)

#Women's
full_data_w = combine_data_results(combined_w, tourney_results_w)


print(full_data_m.columns)

Index(['Season', 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'margin',
       'result', 'NumOT', 'seed_A', 'ap_preseason_rank_A',
       'ap_pretournament_rank_A', 'avg_massey_rank_A', 'avg_score_A',
       'avg_opp_score_A', 'avg_margin_A', 'avg_poss_A', 'avg_eff_A',
       'avg_opp_eff_A', 'avg_fg_per_A', 'avg_fg_a_per_A', 'avg_thr_a_per_A',
       'avg_to_per_A', 'avg_blk_per_A', 'avg_foul_rec_per_A', 'avg_foul_per_A',
       'avg_or_per_A', 'avg_dr_per_A', 'avg_opp_fg_a_per_A',
       'avg_opp_fg_per_A', 'avg_opp_to_per_A', 'avg_thr_per_A', 'ft_per_A',
       'seed_B', 'ap_preseason_rank_B', 'ap_pretournament_rank_B',
       'avg_massey_rank_B', 'avg_score_B', 'avg_opp_score_B', 'avg_margin_B',
       'avg_poss_B', 'avg_eff_B', 'avg_opp_eff_B', 'avg_fg_per_B',
       'avg_fg_a_per_B', 'avg_thr_a_per_B', 'avg_to_per_B', 'avg_blk_per_B',
       'avg_foul_rec_per_B', 'avg_foul_per_B', 'avg_or_per_B', 'avg_dr_per_B',
       'avg_opp_fg_a_per_B', 'avg_opp_fg_per_B', 'avg_opp_to_

In [41]:
### Unique Data Ideas


#Sets OT to 0.75 or 0.25

#OT Stat adjustment

In [42]:
### Simplyfying

#Men's
full_data_m = (
    full_data_m.assign(seed_dif = full_data_m['seed_A'].astype(int) - full_data_m['seed_B'].astype(int))
            .assign(massey_rank_dif = lambda x: x['avg_massey_rank_A'] - x['avg_massey_rank_B'])
            .assign(avg_margin_dif = lambda x: x['avg_margin_A'] - x['avg_margin_B'])
            .assign(thr_a_per_dif = lambda x: x['avg_thr_a_per_A'] - x['avg_thr_a_per_B'])
            .assign(tempo_pred = lambda x: (x['avg_poss_A'] + x['avg_poss_B'])/2)
            .assign(pred_fg_per_dif = lambda x: (x['avg_fg_per_A'] + x['avg_opp_fg_per_B'])/2 - (x['avg_fg_per_B'] + x['avg_opp_fg_per_A'])/2)
            .assign(pred_or_per_dif = lambda x: (x['avg_or_per_A'] + x['avg_dr_per_B'])/2 - (x['avg_or_per_B'] + x['avg_dr_per_A'])/2)
)

simple_data_m = full_data_m[['Season', 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'margin', 'result', 'NumOT', 'seed_dif', 'massey_rank_dif', 'avg_margin_dif', 'avg_eff_A',
       'avg_opp_eff_A', 'avg_eff_B', 'avg_opp_eff_B', 'avg_thr_per_A', 'ft_per_A', 'avg_fg_per_A', 'avg_fg_a_per_A', 'avg_thr_a_per_A',
       'avg_to_per_A', 'avg_blk_per_A', 'avg_opp_fg_a_per_A', 'avg_opp_fg_per_A', 'avg_opp_to_per_A', 'avg_thr_per_B', 'ft_per_B', 'avg_fg_per_B',
       'avg_fg_a_per_B', 'avg_thr_a_per_B', 'avg_to_per_B', 'avg_blk_per_B', 'avg_opp_fg_a_per_B', 'avg_opp_fg_per_B', 'avg_opp_to_per_B', 'thr_a_per_dif', 'tempo_pred', 'pred_fg_per_dif', 'pred_or_per_dif']]

#Women's
full_data_w = (
    full_data_w.assign(seed_dif = full_data_w['seed_A'].astype(int) - full_data_w['seed_B'].astype(int))
            .assign(avg_margin_dif = lambda x: x['avg_margin_A'] - x['avg_margin_B'])
            .assign(thr_a_per_dif = lambda x: x['avg_thr_a_per_A'] - x['avg_thr_a_per_B'])
            .assign(tempo_pred = lambda x: (x['avg_poss_A'] + x['avg_poss_B'])/2)
            .assign(pred_fg_per_dif = lambda x: (x['avg_fg_per_A'] + x['avg_opp_fg_per_B'])/2 - (x['avg_fg_per_B'] + x['avg_opp_fg_per_A'])/2)
            .assign(pred_or_per_dif = lambda x: (x['avg_or_per_A'] + x['avg_dr_per_B'])/2 - (x['avg_or_per_B'] + x['avg_dr_per_A'])/2)
)

simple_data_w = full_data_w[['Season', 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'margin', 'result', 'NumOT', 'seed_dif', 'avg_margin_dif', 'avg_eff_A',
       'avg_opp_eff_A', 'avg_eff_B', 'avg_opp_eff_B', 'avg_thr_per_A', 'ft_per_A', 'avg_fg_per_A', 'avg_fg_a_per_A', 'avg_thr_a_per_A',
       'avg_to_per_A', 'avg_blk_per_A', 'avg_opp_fg_a_per_A', 'avg_opp_fg_per_A', 'avg_opp_to_per_A', 'avg_thr_per_B', 'ft_per_B', 'avg_fg_per_B',
       'avg_fg_a_per_B', 'avg_thr_a_per_B', 'avg_to_per_B', 'avg_blk_per_B', 'avg_opp_fg_a_per_B', 'avg_opp_fg_per_B', 'avg_opp_to_per_B', 'thr_a_per_dif', 'tempo_pred', 'pred_fg_per_dif', 'pred_or_per_dif']]


In [43]:
### Save Data
simple_data_m.to_csv("train_data_m.csv", index=False)
simple_data_w.to_csv("train_data_w.csv", index=False)